In [1]:
import pandas as pd
import glob
import os
from pathlib import Path

# 1. Path to your 'request' folder inside data_files
path = Path.cwd() / 'data_files' / 'request'
all_files = glob.glob(str(path / "*.csv"))

print(f"Found {len(all_files)} precipitation files. Merging ALL columns...")

li = []

# 2. Read and append each file, bringing in ALL columns this time
for filename in all_files:
    try:
        # Removed the 'usecols' parameter to import the entire dataset
        df = pd.read_csv(filename)
        li.append(df)
    except Exception as e:
        print(f"Error reading {filename}. Error: {e}")

if not li:
    raise RuntimeError("No files were successfully read. Double-check your folder path!")

# 3. Combine everything into one giant DataFrame
df_precip = pd.concat(li, axis=0, ignore_index=True)

# 4. Rename ONLY the core spatial/temporal columns so our join script still works.
# All other columns (temp_2m, relative_humidity, soil_moisture, etc.) keep their original names.
CORE_MAP = {
    'station': 'STATION',
    'time': 'DATE',
    'latitude [degrees_north]': 'LATITUDE',
    'longitude [degrees_east]': 'LONGITUDE'
}
df_precip = df_precip.rename(columns=CORE_MAP)

# Convert DATE to datetime for spatial and temporal joining later
print("Standardizing timestamps...")

# 1. Strip the ambiguous timezone text off the end of the strings
df_precip['DATE'] = df_precip['DATE'].str.replace(' EDT', '', regex=False).str.replace(' EST', '', regex=False)

# 2. Convert to a naive datetime object
df_precip['DATE'] = pd.to_datetime(df_precip['DATE'])

# 3. Explicitly localize to NY time, then convert to UTC to perfectly match your flood data
# Note: ambiguous='NaT' prevents crashes during the 1-hour Daylight Saving "fall back" overlap
df_precip['DATE'] = df_precip['DATE'].dt.tz_localize(
    'America/New_York', 
    ambiguous='NaT', 
    nonexistent='NaT'
).dt.tz_convert('UTC')

# Save as a highly compressed Parquet file
df_precip.to_parquet('nyc_precipitation_master.parquet', index=False)

print(f"\nSuccess! Master file saved with {len(df_precip):,} rows.")
print("\nAvailable Data Variables for Future Graphics:")
for col in df_precip.columns:
    if col not in CORE_MAP.values(): 
        print(f" - {col}")

Found 0 precipitation files. Merging ALL columns...


RuntimeError: No files were successfully read. Double-check your folder path!